# 第9回 演習：パイプライン

## 前回からの接続

第8回では、心臓病データで「病気かどうか」の確率を出し、混同行列（confusion matrix）としきい値で見逃しと空振りを測り分けました。そのとき工程は、欠損を埋め・標準化（standardization）し・学習し・確率を出し・しきい値で切る、と長く伸びていて、`pipe()` という自前の小さな関数でようやく一本に保っていました。

今日は、その「一本にまとめる」こと自体が主題です。列によってふさわしい前処理（preprocessing）が違う（数値は標準化、カテゴリはワンホット）ところまで引き受けるのが `ColumnTransformer`、そこからモデルまで束ねるのが `Pipeline`。手作業では守りきれない順序を部品の構造として固め、第4回で見たリーク（leakage）を起こしようがなくします。データは第8回と同じ心臓病、変えるのは見る目——組み立ての目です。

## 今日の分析目標

**前処理から評価までを、ミスなく再現できる一本の流れにしたい。**

この演習では、ColumnTransformer で列ごとの前処理をまとめ、Pipeline でモデルまで束ね、GridSearchCV で設定を選ぶ——その一本の流れを、心臓病データで自分の手で組みます。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。


## 学習ゴール

この回を終えると、次のことができるようになります。

- `ColumnTransformer` で列ごとに前処理を割り当てる理由を説明でき、13列がワンホット（one-hot）で28列に開くまでを追える
- `Pipeline` がリークを構造的に防ぐ仕組みを、「各分割の内側で前処理を学習し直す」という言葉で説明できる
- リークの害の大きさは前処理の中身しだいだと、このデータでの実測（差はほぼ 0.000）を根拠に言える
- `GridSearchCV` で `model__C` を交差検証で選べ、設定選びにテストを使ってはいけない理由を言える
- 交差検証スコアとテストスコアのズレを読み、`best_score_` が少し甘い理由と、格子の端が選ばれたときの危険を説明できる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/heart.csv')
num = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
X, y = df[num + cat], df['target'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'訓練 {len(X_tr)}件 / テスト {len(X_te)}件')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {1, 2, 3}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. ColumnTransformer + Pipeline を組む

数値には「補完（imputation）→標準化」、カテゴリには「補完→ワンホット」を割り当て、モデルまで一本にします。

### 深掘り：列ごとに前処理を分け、一本に束ねる——そして「なぜリークが構造的に防げるのか」

この回の主役は、**前処理を交差検証の内側で正しく回す**ことです。まず道具立てを二段で押さえ、そのうえで「なぜ束ねるとリーク（テスト情報の漏れ）が構造的に防げるのか」という、いちばん大事な理屈に踏み込みます。

**① なぜ列ごとに処理を変えるのか（ColumnTransformer）**　この心臓病データ（303人・心臓病139／健康164）の 13 列は、性質の違う二種類が混ざっています。年齢・血圧・コレステロール・最大心拍・ST低下（`oldpeak`）の **数値5列**と、性別・胸痛タイプ `cp`・空腹時血糖 `fbs`・心電図 `restecg`・運動誘発狭心症 `exang`・傾き `slope`・主要血管数 `ca`・欠陥 `thal` の **カテゴリ8列**です。数値には「中央値で欠損を埋める → 標準化して土俵を揃える」、カテゴリには「最頻値で埋める → ワンホットで 0/1 の列に開く」——**ふさわしい前処理が列ごとに違う**。ふつうの Pipeline は全列に同じ処理をかけてしまうので、この使い分けができません。`ColumnTransformer` は「この列群にはこの前処理、あの列群にはあの前処理」と割り当て、結果を横に連結する道具です。実際、この節のコードセルで組む `pre` に `X_tr` を通すと、カテゴリ8列がワンホットで 23 本のダミー列に開き、数値5列と合わせて **13列 → 28列** に広がります（`cp`・`ca` は4値、`restecg`・`slope`・`thal` は3値…と、値の種類ぶんだけ 0/1 列が増える）。この列数の管理を手作業でやると取り違えのもとですが、部品に任せれば自動です。なお `OneHotEncoder(handle_unknown='ignore')` を付けているのは、学習側になかったカテゴリが検証・テスト側に出てきたときに、エラーで止めず「その値はどのダミー列も 0」として静かに処理させるため。分割のしかたによっては珍しいカテゴリが片側にしか現れないので、この保険が効きます。

**② なぜ一本に束ねるのか（Pipeline）**　`Pipeline([('pre', pre), ('model', ...)])` は、前処理からモデルまでを**一つの推定器**に見せます。`fit` を一度呼べば「補完 → 標準化／ワンホット → 学習」が正しい順序で走り、`predict` を呼べば**同じ変換が同じ順序で**適用される。順序を手で書かないので、順序ミスがそもそも起きません。コードが短くなるのは副次的な利点で、本当の狙いは次の③です。

**③ 核心：交差検証の「各分割の内側」で前処理を閉じる**　リークは、**本来知り得ないテスト側の情報が、こっそり学習に混ざる**事故です。交差検証（cross-validation）は手元データを5つに分け、4つで学習・1つで検証…を5回まわして性能を測ります。ここで前処理を**うっかり全データで先に学習**してしまうと、こうなります——標準化に使う平均・標準偏差（standard deviation）、補完に使う中央値・最頻値、ワンホットが知る「カテゴリの顔ぶれ」が、**その回の検証用に取り分けた1/5の行まで覗いて**計算されてしまう。検証データは「まだ見ていない未知データ」の代役なのに、その素性を前処理がすでに知っている。これがリークです。

具体的に一周を追うと分かります。いま5分割の1回目で「fold1 を検証、fold2〜5 で学習」とします。標準化の平均は fold2〜5 だけから計算し、その平均で fold1 を引き算する——これが正しい姿。ところが全データで先に標準化していると、平均の中に fold1 の 60 人ぶんが混ざっており、**fold1 は「自分も混ぜて作った物差し」で測られる**。2回目は fold2 が検証に回るのに、その物差しは1回目に fold2 を見て作られている……と、どの回でも検証データが前処理を通じて学習に顔を出してしまう。正しくは、**5回それぞれの中で、学習用4/5だけから前処理を学習し直し、検証用1/5にはそれを適用（transform）するだけ**にしなければなりません。

`Pipeline` を `cross_val_score` や次節の `GridSearchCV` に渡すと、この「各分割の内側で前処理を fit し直す」が**自動**になります。分割のたびに Pipeline 全体が学習用ぶんだけで `fit` され、検証用ぶんには `transform` だけ。手で書けば「毎回、前処理を学習し直す」面倒な処理を自分で組むことになり、一行書き忘れれば即リーク。それを部品が構造的に封じてくれる——これが「Pipeline はリークを防ぐ」の中身です。

**④ この実データで裏を取る（正直な結果）**　では、全データで前処理を先に学習する“横着”は、この心臓病データでどれだけスコアを水増しするのか。この節のあと、Pipeline を組んだ直後のセルで実際に比べます——`pre` を `X_tr` 全体で先に `fit` してから交差検証した「リークあり」と、Pipeline を交差検証の内側で回した「正しい」やり方。その出力のとおり、**どちらも CV AUC = 0.898**（`C=0.1`）で、差はほぼ **0.000** でした。拍子抜けするほど差がありません。理由は、ここの前処理が**穏やかだから**です。標準化は各列を定数で割り引くだけ、中央値・最頻値補完も欠損はわずか6セル（`ca` 4件・`thal` 2件）で、AUC（順位づけの良さ。第8回演習を参照）はこうした穏やかな変換にほとんど動じません。**だから安全、ではありません**。差が出ないのは今回の前処理と数字の巡り合わせにすぎず、Pipeline は**タダで正しさを保証**してくれるのだから使わない手はない、というのが正しい読み方です。

前処理が攻めた瞬間、横着は牙をむきます。たとえば「無関係な500個のノイズ列から、正解ラベルを見ながら“効く”20列を選ぶ」ことを全データでやってから交差検証すると、CV AUC は本来ほぼ当てずっぽうの **0.58** であるべきところ、**0.82** まで偽物のスコアに水増しされます（正しく各分割の内側で選べば 0.58 に戻る）。変数選択・次元削減（dimensionality reduction）・目的変数（response variable）を使う変換（ターゲットエンコーディングなど）が入るほど、この漏れは大きくなる。だから「小さいから大丈夫」ではなく「**構造で閉じておく**」のです。

**⑤ つまずきの鉄則：`fit` をテストに適用しない**　言い換えれば、前処理の**素性（平均・中央値・カテゴリ一覧）を決める `fit` は、学習データだけで行う**。テストや検証データに対しては、その決まった変換を当てはめる `transform` **だけ**を行う。`scaler.fit(X_test)` と書いた瞬間に、テストの平均でテストを標準化してしまい、テストが「未知」でなくなります。Pipeline を使えば `fit` は学習ぶん、`transform` はテストぶん、と自動で振り分けられるので、この一線を越えようがありません。この「fit は学習だけ／test には transform だけ」は、第3回・第5回演習で繰り返してきた「テストは最後まで隠す」の、前処理版の言い換えです。分割の作り方（ブートストラップや交差検証といったリサンプリング）ごとに、どんな量が漏れるとどれだけ楽観的な推定になるのかを確率の言葉で書き下す形式的な議論は、リサンプリング法を扱う統計計算の専門書に譲ります。ここでは「各分割の内側で前処理を閉じる」という骨格をつかんでください。

**全体像：一本の流れとして**　ここで組む部品は、教師あり学習の**完成形ワークフロー**の中核です——①データを分割（済み）→ ②`ColumnTransformer`＋`Pipeline` で列ごとの前処理とモデルを一本化 → ③次節の `GridSearchCV` で学習データの内側だけで設定を最適化（optimization） → ④選んだ設定をテストで一度だけ評価 → ⑤納得できたら全データで学習し直して本番へ。手作業なら順序ミスとリークが忍び込む①〜⑤の継ぎ目を、Pipeline が構造的に塞ぎ、GridSearchCV が探索を自動化する。バラバラだった前処理・モデル・評価が、**誰がいつ回しても同じ結果になる一本の流れ**にまとまる——これがこの回の目標そのものです。


In [ ]:
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat)])
pipe = Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=5000))])
print('Pipeline を組みました')

In [ ]:
# §④の裏取り：前処理を「全データで先に学習」しても、交差検証スコアは変わるか
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
# 正しい：Pipeline を交差検証の内側で回す（前処理も各分割の学習ぶんだけで学習し直す）
correct = cross_val_score(Pipeline([('pre', clone(pre)),
                                    ('model', LogisticRegression(C=0.1, max_iter=5000))]),
                          X_tr, y_tr, cv=5, scoring='roc_auc').mean()
# リークあり：pre を X_tr 全体で先に fit してから交差検証（検証ぶんまで前処理に覗かせる）
X_all = clone(pre).fit_transform(X_tr)
leaky = cross_val_score(LogisticRegression(C=0.1, max_iter=5000),
                        X_all, y_tr, cv=5, scoring='roc_auc').mean()
print(f'正しい（Pipeline を交差検証の内側）: CV AUC = {correct:.4f}')
print(f'リークあり（全データで先に前処理） : CV AUC = {leaky:.4f}')
print(f'差: {leaky - correct:+.4f}  ← 穏やかな前処理なのでほぼ 0。だが Pipeline なら常にタダで安全')

In [ ]:
# 図：Pipeline が束ねるのはどこか（②「前処理とデータ解析」の中身を手順に開く）
from matplotlib.patches import FancyBboxPatch
from sklearn.base import clone
n_in = X_tr.shape[1]                                # 前処理に入る列数
n_out = clone(pre).fit_transform(X_tr).shape[1]     # ワンホットで開いたあとの列数

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
acc, gray = '#6741d9', '#8a8f98'
steps = [('手順1\n分割', gray, 0.02),
         (f'手順2 前処理\nColumnTransformer\n{n_in}列 → {n_out}列', acc, 0.28),
         ('手順3 モデル\nロジスティック回帰', acc, 0.53),
         ('手順4 評価\n交差検証・テスト', gray, 0.79)]
w, h, y0 = 0.19, 0.34, 0.26
for label, color, x in steps:
    ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle='round,pad=0.008',
                                facecolor=color, edgecolor='none'))
    ax.text(x + w / 2, y0 + h / 2, label, ha='center', va='center',
            color='white', fontsize=11)
for x0 in [0.21, 0.47, 0.72]:
    ax.annotate('', xy=(x0 + 0.055, y0 + h / 2), xytext=(x0, y0 + h / 2),
                arrowprops=dict(arrowstyle='->', color='#666', lw=2))
ax.add_patch(FancyBboxPatch((0.265, 0.20), 0.485, 0.46, boxstyle='round,pad=0.008',
                            facecolor='none', edgecolor=acc, lw=2, linestyle='--'))
ax.text(0.508, 0.73, 'Pipeline（前処理とモデルを一つの推定器に束ねる）',
        ha='center', fontsize=12, color=acc)
ax.text(0.5, 0.07, '手順1・4 は Pipeline の外側。分割のたびに、破線の中がまるごと学習し直される',
        ha='center', fontsize=11, color='#555')
plt.tight_layout(); plt.show()


**この図の読み方**　紫の二つの箱が `Pipeline` の中に入る手順、灰色の二つはその外に残る手順です。手順2 の `ColumnTransformer` は 13 列を受け取り、カテゴリ8列をワンホットで開いて **28列** の数値表にしてから、手順3 のロジスティック回帰（logistic regression）へ渡します（この列数は図のセルで実際に数えた値で、この節のはじめの深掘り①で見た「13列 → 28列」と同じです）。破線の枠が `Pipeline` の輪郭で、束ねているのは手順2と手順3の二つだけ——分割（手順1）と評価（手順4）は、その外側にあります。

外側にあることが、そのままリーク防止の話につながります。交差検証は手順1の分け直しと手順4の採点を5回くり返しますが、そのたびに破線の中がまるごと学習し直されるので、前処理が検証用に取り分けた 1/5 を覗く隙がありません。この節の TODO① で使う `GridSearchCV` は、その5回を候補5つぶん、つまり 25 回まわします。破線が一つの部品として閉じているかぎり、25 回のどれでもリークは起きない——それがこの図の言いたいことです。


### TODO①：GridSearchCV で C を最適化する

`param_grid` に `model__C` の候補（例：0.01, 0.1, 1, 10, 100）を用意し、GridSearchCV（cv=5, scoring='roc_auc'）で最良のCを見つけてください。選ばれたCと、交差検証での最良AUCを表示しましょう。

### 深掘り：ハイパーパラメータと GridSearchCV——25回の学習で「山の頂上」を選ぶ

**係数と、ハイパーパラメータ（hyperparameter）は別物**　モデルの数字には二種類あります。ひとつは**学習で自動的に決まる係数**（各検査値の重み）。もうひとつは、**人が学習前に決める設定＝ハイパーパラメータ**で、ここでの `C` がそれです。`C` はロジスティック回帰の正則化（regularization）の強さで、第7回演習で見た手綱と地続き——ただし**逆数**の関係で、`C` が**小さいほど正則化は強く**（係数をぎゅっと抑える）、大きいほど罰金は軽く自由なモデルになります。係数はデータが決めますが、`C` はデータから自動では決まりません。だから、いくつか候補を試して**いちばん良いものを選ぶ**必要があります。

**なぜ勘でも、テストでもダメか**　`C` を勘で決めては、せっかくデータに語らせた意味が薄れます。かといって、いろいろな `C` をテストで試して良かったものを採れば、テストを何度も覗くことになり——それ自体がリークです（第3回演習の「テストは最後に一度だけ」に反する）。正しくは、**学習データの内側だけで交差検証**して選ぶ。これを自動化したのが `GridSearchCV` です。

**GridSearchCV が内側でしていること**　`GridSearchCV(pipe, {'model__C':[0.01,0.1,1,10,100]}, cv=5, scoring='roc_auc')` は、5つの `C` 候補それぞれを5分割の交差検証で採点し、平均スコアが最良の候補を選びます。内側では **5候補 × 5分割 = 25回** の学習が走っています。しかも渡しているのは素のモデルではなく **Pipeline** なので、その25回すべてで、前処理も各分割の学習ぶんだけから学習し直されます（前節③の仕組みがそのまま効く）。つまり**設定探しの最中もリークが起きない**——ここが Pipeline と GridSearchCV を組み合わせる真価です。採点基準の `scoring='roc_auc'` は AUC（第8回演習で扱った、しきい値に依らない順位づけの良さ）で、正解率よりクラスの偏りに強い指標です。

**`model__C` という名指しの作法**　候補の鍵 `model__C` は、アンダーバー二つの**前がステップ名、後ろがその設定名**。「`model` ステップの `C`」を指します。同じ流儀で `pre__num__imp__strategy`（数値の補完を平均か中央値か）まで探索対象にできる——モデルの設定だけでなく前処理の設定すらデータに選ばせられる、というのがこの命名規則の狙いです。ステップに名前を付けておいたのは、このためでした。

**選んだ後は、そのまま使える**　`GridSearchCV` は探すだけでなく、`best_estimator_` に**最良設定で学習し直した Pipeline** を用意してくれます。これ一つで新しい患者データを渡せば、補完・標準化・ワンホット・予測までまとめて走る。そして評価がすべて済んだあとの本番運用では、テスト用に取り分けた分も**惜しみなく合流させ、手元の全データで学習し直す**のが定石です（評価は終わったので、もう分けておく必要がない）。探索・評価・仕上げが一続きになるのが、この道具立ての気持ちよさです。

**発展：総当たりが重いなら「くじ引き」に**　候補が増えると学習回数は掛け算で爆発します（設定Aが5通り × 設定Bが5通り × 5分割 = 125回）。そこで、全部を試さず候補空間から**ランダムに何個か**引く `RandomizedSearchCV` という手があります。実際、この `C` を対数一様分布から10個引かせると `C ≈ 0.34`（CV AUC 0.896）を見つけ、格子で選ばれた `C=0.1`（0.898）とほぼ同等の谷にちゃんと着地します。候補が広いときは「総当たり（grid）」より「くじ引き（random）」が費用対効果で勝つことが多い、と覚えておくと使い分けが効きます。まずは今の格子で `C` を探し、`best_params_` と `best_score_` を表示してみましょう。

**発展：粗く探してから絞る／モデルの種類も比べる**　計算が爆発するなら、段取りで抑えます。最初は `[0.01, 0.1, 1, 10, 100]` のように**桁で粗く**当たりをつけ、良さそうな範囲（今回なら 0.1 付近）だけを `[0.05, 0.1, 0.2, 0.5]` と**細かく**探り直す——この二段構えが定石です。さらに `GridSearchCV` は `C` のような数値設定だけでなく、**モデルの種類そのもの**も候補にできます。ロジスティック回帰・決定木（decision tree）・ランダムフォレスト（random forest）を同じ交差検証の土俵に並べれば、「どのモデルがこの問題に向くか」まで同じ手続きで、勘ではなくデータに選ばせられます。


In [ ]:
# TODO: param_grid を用意し、GridSearchCV で最良の C を探して、best_params_ と best_score_ を表示してください
# ヒント: GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc') を作って fit(X_tr, y_tr)
# ヒント: 設定名は 'model__C'（Pipeline の model ステップの C）
...

## 2. テストで最終評価

TODO①で選んだ設定を、テストで評価します。設定選びにテストを使っていないので、これが「本番の成績」です。

### 深掘り：交差検証スコアとテストスコアのズレの読み方——`best_score_` は少し甘い

**なぜテストで測り直すのか**　`GridSearchCV` は学習データの内側だけで `C` を選び、テストには一切触れていません。だからテストで測った値は、設定選びに使っていない**まっさらな本番の成績**です。下のセルを実行すると、この心臓病データでは **交差検証 AUC 0.898 / テスト AUC 0.965** と出ます。

**テストのほうが高い？——それはリークではない**　交差検証より**テストが高い**と面食らうかもしれませんが、これはリークの兆候ではありません。テストはわずか **61人**しかなく、一発勝負の推定値は上にも下にも大きく振れます。交差検証の各分割のばらつき（次節で見る帯）は AUC で ±0.04〜0.05 あり、テスト 0.965 はその揺らぎの上振れの範囲にすぎません。**警戒すべきは逆で、テストが交差検証より大きく“低い”とき**——それは候補を選びすぎて交差検証スコアに合わせ込んだ（ハイパーパラメータの過学習）サインです。今回はテストが下振れしていないので、選びすぎの心配は小さい、と読めます。

**`best_score_` はちょっと甘い、という注意**　`best_score_`（0.898）は「5候補の中で最も良かったスコア」です。たくさんの候補から最大値を拾う操作は、まぐれの上振れを拾いやすく、**選択そのものの分だけ楽観的**になりがちです。だから `best_score_` を「このモデルの実力」と鵜呑みにせず、独立したテストで確かめる——それが本節の役目です。

**発展：選ぶ手続きごと評価する「ネスト交差検証」**　「`C` を選ぶ手続き自体は、どれくらいの実力なのか」を偏りなく測りたいなら、**ネストした交差検証（nested CV）**があります。外側の分割で本当のテスト役を取り分け、その内側でだけ `GridSearchCV` に `C` を選ばせる——これを外側の各分割で繰り返す二重構造です。選択を毎回“内側”に閉じ込めるので、`best_score_` の甘さが入りません。このデータで回すと **ネスト CV AUC ≈ 0.906**。楽観的な `best_score_`（0.898）とほぼ同じで、この選択手続きが安定していることを裏づけます（差が大きく開けば選びすぎを疑う）。ネスト CV は計算が重い代わりに、「モデル選択を含めた手続きの汎化性能（generalization performance）」を正直に見積もる、最も堅い方法です。どんな場面でネストを省くとどれだけ楽観に転ぶのかを数値実験で突き合わせた、モデル選択と性能評価の比較研究の論文が、深追いの入口になります。

**「選びすぎ」は具体的にどう見えるか**　もし候補を数百・数千と増やし、そのどれかがまぐれで交差検証スコアを跳ね上げたとします。すると `best_score_` は立派な数字になるのに、テストではガクッと落ちる。この**交差検証は良いのにテストが悪い**という食い違いこそ、ハイパーパラメータの過学習（overfitting）の顔です。予防は単純で、候補をむやみに広げない・最後は必ず独立したテスト（あるいは上のネスト CV）で答え合わせをする、の二点。「くじをたくさん引けば、まぐれ当たりも出やすい」——第5回演習の交差検証の教訓が、設定選びの層でもそのまま効いています。


In [ ]:
param_grid = {'model__C': [0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc').fit(X_tr, y_tr)
print(f'選ばれた C: {gs.best_params_["model__C"]}')
print(f'交差検証 AUC: {gs.best_score_:.3f}')
print(f'テスト AUC : {gs.score(X_te, y_te):.3f}')

### TODO②：C と性能の関係を図にする

`gs.cv_results_['mean_test_score']` を使って、C（対数軸）と交差検証AUCの関係を折れ線で描いてください。山の頂上が、選ばれたCです。

### 深掘り：C と性能の曲線を読む——「山の頂上」と、端が選ばれたときの警告

`gs.cv_results_['mean_test_score']` には、候補ごとの交差検証 AUC が候補の並び順（0.01, 0.1, 1, 10, 100）で入っています。この心臓病データでは順に **0.879 / 0.898 / 0.894 / 0.890 / 0.890**。対数軸（`semilogx`）で折れ線にすると、`C=0.01`（正則化が強すぎ＝係数を締めすぎ）の左端で 0.879 と一段低く、`C=0.1` で **0.898** の頂上をつけ、そこから `C` を大きく（正則化を弱く）していくとゆるやかに下がって 0.890 前後で頭打ち——**ゆるい山型**になります。GridSearchCV はこの頂上、`C=0.1` を選びました。これは第7回演習で見た「強すぎず弱すぎず、ちょうどよい手綱」を探すU字（を上下反転した山）そのもので、正則化の話がハイパーパラメータ探索としてそのまま現れています。

差はどれも 0.879〜0.898 と小さく、しかも各点には交差検証の分割ごとのばらつき（`std_test_score` ≈ 0.04〜0.05）が乗っています。TODO②の図では点を打つだけですが、山の高低差（約0.02）が帯の幅（約0.05）より小さいことも意識すると、「頂上は 0.1 あたりだが、0.1〜100 のどれもそう大差はない」という現実的な読みができます。ハイパーパラメータ探索は、一点を神聖視するより**なだらかな高原のどこに乗っているか**を掴む作業だ、という感覚が役立ちます。

**つまずき：端が選ばれたら範囲を疑う**　もし最良が候補の**端**（`0.01` や `100`）だったら要注意です。それは「本当の頂上はもっと外側にあるのに、格子が狭くて途中で切れている」サインで、範囲を広げて探し直すべきです。今回は端ではなく内側の `C=0.1` で山を作っているので、探索範囲は妥当だった、と確認できます。選ばれた値がグリッドの端でないか——`best_params_` を出したら必ず目で確かめる習慣をつけてください。


In [ ]:
# TODO: 横軸C（対数）、縦軸 CV AUC の折れ線グラフを描いてください
# ヒント: Cs = [0.01, 0.1, 1, 10, 100]、plt.semilogx(Cs, gs.cv_results_['mean_test_score'], 'o-')
...

## 目標に答えられたか

- 今日の目標は「前処理から評価までを、ミスなく再現できる一本の流れにしたい」でした
- TODO①で、選ばれたCはいくつでしたか？ 交差検証AUCは？
- テストAUCは、交差検証AUCと比べてどうでしたか？（大きく下がっていなければ、選びすぎていないサイン）
- この Pipeline は、なぜリークを構造的に防げるのでしょうか？（各分割の中で前処理がどう学習されるか）
- 補完・標準化・ワンホットを手作業で書くのと比べて、Pipeline の利点は何でしたか？

## 今日の要点

- 列によってふさわしい前処理は違う。だから全列に同じ処理をかけず、`ColumnTransformer` で「この列群にはこれ」と割り当てる。13列 → 28列という列数の管理も部品が引き受ける
- `Pipeline` の狙いはコードが短くなることではなく、前処理 → モデルの順序を構造で固定すること。順序を手で書かなければ、順序ミスは起こしようがない
- リークを防ぐ要は「各分割の内側で前処理を閉じる」こと。素性（平均・中央値・カテゴリの顔ぶれ）を決める `fit` は学習ぶんだけ、検証やテストには `transform` だけを当てる
- このデータではリークありでも交差検証 AUC は 0.898 のままで、差はほぼ 0.000 だった。害の大きさは前処理の攻めぐあい次第——「小さかったから安全」ではなく「タダで安全なら構造で閉じる」と読む
- ハイパーパラメータは学習では決まらない。だから候補を並べ、学習データの内側の交差検証で選ぶ。テストを見ながら選べば、それ自体がリークになる
- `best_score_` は候補の最大値なので、まぐれの上振れを拾って少し甘い。独立したテストで測り直し、テストが大きく下振れしていないかを確かめる
- 選ばれた値が格子の端なら探索範囲を疑う。今回の頂上は `C=0.1` だが、候補間の高低差 0.02 は分割ごとのばらつき（±0.04〜0.05）より小さく、一点を神聖視する話ではない


## 次回へ

分割・補完・標準化・ワンホット・交差検証・分類（classification）、そしてそれらを束ねる `Pipeline` と、設定を選ぶ `GridSearchCV`。第2回から一つずつ集めてきた道具が、これでひととおりそろいました。ここまでの各回は、地図のどこか一か所を取り出して練習する回でした。

次回は、取り出すのをやめて一周します。目標を決め、データを理解し、前処理を設計し、学習して、評価して、解釈して報告する——その通しを自分の手で最後まで回す総合演習です。データは心臓病から南極のペンギンに移り、当てるのは二択ではなく3種のどれか。使う部品はすべて既習で、難しいのは部品ではありません。順序を守って一周を閉じきることです。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節の `pipe` は、モデルがロジスティック回帰でした。今度はモデルだけを差し替えます。
`model` ステップを `RandomForestClassifier(random_state=42)` にした Pipeline を作り、`param_grid` を `{'model__max_depth': [3, 5, None], 'model__n_estimators': [100, 300]}` にして、GridSearchCV（cv=5, scoring='roc_auc'）で設定を探してください。選ばれた設定と、交差検証 AUC を表示しましょう。
`max_depth` は木の深さの上限（`None` は制限なし）、`n_estimators` は木の本数です。前処理 `pre` はそのまま使えます。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`from sklearn.ensemble import RandomForestClassifier` を書き、1節の `Pipeline([('pre', pre), ('model', ...)])` の `...` を差し替えるだけです。2節の `GridSearchCV(...).fit(X_tr, y_tr)` はそのまま。6候補 × 5分割 = 30回の学習なので、10秒ほどかかります。

</details>


### 応用②（判断）

応用①で選ばれたランダムフォレストのテスト AUC は、2節のロジスティック回帰のテスト AUC より**上ですか、下ですか**。「上」か「下」で答えてください。両方の値も小数第3位まで表示しましょう。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

2節と同じく、GridSearchCV の `.score(X_te, y_te)` がテスト AUC です。ロジスティック回帰は `gs`、ランダムフォレストは応用①で作ったものを使います。

</details>


### 応用③（解釈）

応用②で見たとおり、2つのモデルのテスト AUC の差は小さく、交差検証の分割ごとのばらつき（±0.04〜0.05）の中に収まります。交差検証では順位が逆でした。それ自体がばらつきの大きさの証拠です。精度が同程度なら、この心臓病データを使った診断支援にはどちらのモデルを勧めますか。説明のしやすさ・学習にかかる時間・前処理の要否（標準化が要るかどうか）の3点を根拠に、医師に向けて3行で書いてください。


（ここに3行程度で書く）


<details><summary>詰まったら</summary>

学習時間は応用①のセルの先頭に `%%time` を付けると測れます。決定木の仲間は大小の順序しか見ないので、標準化しても結果は変わりません。

</details>


## 発展（任意）

### LightGBM を同じ Pipeline に載せる

表形式のデータ（行が人、列が検査値のような表）では、**勾配ブースティング**という手法が、データ分析コンペや実務で最有力とされています。小さな決定木を何百本も順に足していき、前の木の間違いを次の木が直す、という仕組みです。その代表的な実装が `LightGBM` です。

scikit-learn の外のライブラリですが、scikit-learn と同じ `fit` / `predict` の作法で書かれているので、1節の Pipeline の `model` ステップに**そのまま差し替えられます**。前処理は共通のまま、モデルだけを入れ替えられるのが Pipeline の強みです。

ただし LightGBM は設定項目（木の本数・学習率・深さなど）が多く、小さなデータでは簡単に過学習します。だから交差検証で成績を確かめるのは必須です。ここでは1節の `pre` に載せて、ロジスティック回帰と同じ土俵で比べます。


In [ ]:
# Colab には入っていますが、なければインストールする
import importlib.util
if importlib.util.find_spec('lightgbm') is None:
    %pip install -q lightgbm


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

pipe_lgb = Pipeline([('pre', pre),
                     ('model', LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                              random_state=42, verbose=-1))])
cv_lgb = cross_val_score(pipe_lgb, X_tr, y_tr, cv=5, scoring='roc_auc').mean()
pipe_lgb.fit(X_tr, y_tr)
test_lgb = roc_auc_score(y_te, pipe_lgb.predict_proba(X_te)[:, 1])

print('モデル                     交差検証 AUC  テスト AUC')
print(f'ロジスティック回帰 (C=0.1)   {gs.best_score_:.3f}        {gs.score(X_te, y_te):.3f}')
print(f'LightGBM                   {cv_lgb:.3f}        {test_lgb:.3f}')


**読み方**　交差検証 AUC は、ロジスティック回帰が 0.898、LightGBM が 0.872 です。「最有力」のはずの LightGBM が、この心臓病データでは交差検証で 0.026 下回っています。これも分割のばらつきの範囲内ですが、少なくとも上回る根拠はありません。訓練データが 242 人しかなく、200 本の木を足していく方法には少なすぎて過学習しているためと考えられます（下の「試すなら」で学習率を下げると CV AUC が上がるのが、その傍証です）。小さなデータでは、単純なモデルのほうが強いことがよくあります。

テスト AUC は 0.965 と 0.964 で、ほぼ同じです。交差検証では 0.026 の差があったのにテストでは並ぶ——これは 61 人のテストの揺らぎ（2節の深掘りで見た ±0.04〜0.05）の範囲です。交差検証とテストの両方を見て、「このデータでは LightGBM を使う理由はない」と判断するのが正しい読み方です。

コードで変えたのは `model` ステップの1行だけです。前処理・分割・評価はすべて同じなので、比較は公平です。

試すなら、`learning_rate` を 0.01 にしてみてください。1本の木の効きを弱めると過学習が和らぎ、交差検証 AUC が上がります。`GridSearchCV` に `model__learning_rate` を渡して選ばせることもできます。応用①のランダムフォレストの値も、この表に自分で足してみてください。
